# DHS aggregate immunization research

The DHS Program Indicator Data API, The Demographic and Health Surveys (DHS) Program. ICF. Originally funded by the United States Agency for International Development (USAID). Available from https://api.dhsprogram.com/. See fetch_dates for access times. SurveyBridge AI is independent of The DHS Program.

Read limitations.md. Survey-specific comparability is unverified. No interpolation or inference.

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
root = Path.cwd()
manifest = json.loads((root / 'provenance.json').read_text(encoding='utf-8'))
for name, expected in manifest['output_checksums'].items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name
project = json.loads((root / 'project.json').read_text(encoding='utf-8'))
from surveybridge.services.provenance import verify_project
assert verify_project(project)
print('All archived response and output hashes verified')

All archived response and output hashes verified


In [2]:
from surveybridge.services.series import normalize
from surveybridge.models import Indicator
source_records = []
for page in project['pages']:
    if '/data?' not in page['url']:
        continue
    for raw in json.loads(bytes.fromhex(page['body_hex']))['Data']:
        from surveybridge.utils.cache import canonical, digest
        source_records.append(dict(raw=raw, source_url=page['url'], retrieved_utc=page['retrieved_utc'], source_response_hash=page['sha256'], source_record_hash=digest(canonical(raw))))
rebuilt, _ = normalize(source_records, {i['id']: Indicator(**i) for i in project['indicators']})
expected = sorted(project['observations'], key=lambda r: r['source_row_id'])
assert sorted(rebuilt, key=lambda r: r['source_row_id']) == expected
df = pd.DataFrame(json.loads((root / 'observations.json').read_text(encoding='utf-8')))
print('Every normalized observation reproduced from original API records')
df[['source_country_name', 'source_survey_id', 'survey_year_label', 'indicator_id', 'value', 'unit']]

Every normalized observation reproduced from original API records


,source_country_name,source_survey_id,survey_year_label,indicator_id,value,unit
0,Kenya,KE1989DHS,1989,CH_VACC_C_DP3,55.1,Percent
1,Kenya,KE1993DHS,1993,CH_VACC_C_DP3,86.8,Percent
2,Kenya,KE1998DHS,1998,CH_VACC_C_DP3,79.2,Percent
3,Kenya,KE2003DHS,2003,CH_VACC_C_DP3,72.2,Percent
4,Kenya,KE2008DHS,2008-09,CH_VACC_C_DP3,86.4,Percent
...,...,...,...,...,...,...
73,Uganda,UG1995DHS,1995,CH_VACC_C_MSL,59.6,Percent
74,Uganda,UG2000DHS,2000-01,CH_VACC_C_MSL,56.8,Percent
75,Uganda,UG2006DHS,2006,CH_VACC_C_MSL,68.1,Percent
76,Uganda,UG2011DHS,2011,CH_VACC_C_MSL,75.8,Percent


In [3]:
report = json.loads((root / 'comparability_report.json').read_text(encoding='utf-8'))
for finding in report['findings']:
    print(finding['severity'].upper(), finding['explanation'])
by_indicator = json.loads((root / 'comparability_by_indicator.json').read_text(encoding='utf-8'))
for indicator_id, check in by_indicator.items():
    print(indicator_id, 'Descriptive source table; publication comparability unverified')
    if not check['comparison_allowed']:
        print('BLOCKED: review indicator-specific findings before comparison')
        continue
    subset = df[df.indicator_id == indicator_id]
    display(subset[['source_country_name', 'source_survey_id', 'survey_year_label', 'value', 'unit']])
# Keep indicators separate. No annual interpolation, pooling, confidence intervals or significance tests.

CAUTION Repeated cross-sectional surveys do not constitute annual surveillance or causal evidence.
BLOCK Different indicators must not be pooled as equivalent estimates.
CAUTION Catalog allows alternative age populations; survey-specific age equivalence is unverified.
CAUTION Catalog allows alternative age populations; survey-specific age equivalence is unverified.
CAUTION Catalog allows alternative age populations; survey-specific age equivalence is unverified.
CAUTION Source confidence intervals are incomplete or absent.
CAUTION Estimates refer to different survey years and fieldwork periods.
CAUTION Shared catalog metadata does not verify survey-specific questionnaire, vaccine formulation, ascertainment or geographic coverage equivalence.
CH_VACC_C_BCG Descriptive source table; publication comparability unverified


,source_country_name,source_survey_id,survey_year_label,value,unit
26,Kenya,KE1989DHS,1989,59.0,Percent
27,Kenya,KE1993DHS,1993,96.2,Percent
28,Kenya,KE1998DHS,1998,95.9,Percent
29,Kenya,KE2003DHS,2003,87.3,Percent
30,Kenya,KE2008DHS,2008-09,95.6,Percent
31,Kenya,KE2014DHS,2014,96.7,Percent
32,Kenya,KE2022DHS,2022,96.9,Percent
33,Nigeria,NG1990DHS,1990,60.6,Percent
34,Nigeria,NG2003DHS,2003,48.3,Percent
35,Nigeria,NG2008DHS,2008,49.7,Percent


CH_VACC_C_DP3 Descriptive source table; publication comparability unverified


,source_country_name,source_survey_id,survey_year_label,value,unit
0,Kenya,KE1989DHS,1989,55.1,Percent
1,Kenya,KE1993DHS,1993,86.8,Percent
2,Kenya,KE1998DHS,1998,79.2,Percent
3,Kenya,KE2003DHS,2003,72.2,Percent
4,Kenya,KE2008DHS,2008-09,86.4,Percent
5,Kenya,KE2014DHS,2014,89.9,Percent
6,Kenya,KE2022DHS,2022,89.2,Percent
7,Nigeria,NG1990DHS,1990,32.9,Percent
8,Nigeria,NG2003DHS,2003,21.4,Percent
9,Nigeria,NG2008DHS,2008,35.4,Percent


CH_VACC_C_MSL Descriptive source table; publication comparability unverified


,source_country_name,source_survey_id,survey_year_label,value,unit
52,Kenya,KE1989DHS,1989,47.6,Percent
53,Kenya,KE1993DHS,1993,83.8,Percent
54,Kenya,KE1998DHS,1998,79.2,Percent
55,Kenya,KE2003DHS,2003,72.5,Percent
56,Kenya,KE2008DHS,2008-09,85.0,Percent
57,Kenya,KE2014DHS,2014,87.1,Percent
58,Kenya,KE2022DHS,2022,89.0,Percent
59,Nigeria,NG1990DHS,1990,45.6,Percent
60,Nigeria,NG2003DHS,2003,35.9,Percent
61,Nigeria,NG2008DHS,2008,41.4,Percent
